# Projeto MIMIC-III — Análise e Predição de Length of Stay no ICU

**Objetivo:** Analisar os dados clínicos do MIMIC-III, realizar análise estatística e visualização por paciente, e construir um modelo de Machine Learning para prever o tempo de permanência no ICU (Length of Stay).

**Dados principais:** `CHARTEVENTS.csv.gz` (4 GB comprimido) — registos de eventos de pacientes durante a estadia no ICU.

**Estrutura do notebook:**
1. Configuração do ambiente e dependências
2. Carregamento eficiente dos dados com PySpark
3. Análise exploratória e visualização por paciente
4. Construção da variável alvo (LoS)
5. Feature Engineering por janela temporal
6. Pipeline de pré-processamento
7. Treino e validação de modelos
8. Análise de resultados e interpretação
9. Profiling e discussão de performance

---
## 1. Configuração do Ambiente

Instalamos e importamos todas as bibliotecas necessárias. Usamos **PySpark** para processar os ficheiros grandes de forma distribuída, e **scikit-learn / LightGBM** para o pipeline de Machine Learning.

In [ ]:
# Instalar dependências (executar apenas uma vez)
# !pip install pyspark lightgbm xgboost shap optuna pyarrow pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import time
import cProfile
import pstats
import io
import warnings
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

from sklearn.model_selection import GroupShuffleSplit, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import lightgbm as lgb
import xgboost as xgb
import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Variável global para registar tempos de execução
tempos_execucao = {}

print("✅ Bibliotecas importadas com sucesso.")

---
## 2. Inicialização do PySpark

O PySpark permite processar ficheiros de vários GB de forma distribuída, dividindo os dados em partições processadas em paralelo.
Configuramos a sessão com memória adequada para o `CHARTEVENTS.csv.gz` (~4 GB comprimido, ~16 GB descomprimido em memória).

> **Porquê PySpark?** Com pandas puro, carregar 3.6 GB de CSV poderia consumir 10–15 GB de RAM e demorar vários minutos. PySpark faz *lazy evaluation* — só executa operações quando necessário — e processa em paralelo.

In [ ]:
t0 = time.time()

spark = SparkSession.builder \
    .appName("MIMIC-III_LoS_Prediction") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

# Reduzir verbosidade dos logs do Spark
spark.sparkContext.setLogLevel("WARN")

tempos_execucao['spark_init'] = time.time() - t0
print(f"✅ SparkSession iniciada em {tempos_execucao['spark_init']:.2f}s")
print(f"   Versão Spark: {spark.version}")

---
## 3. Definição dos Caminhos dos Ficheiros

Ajusta os caminhos abaixo para o local onde tens os ficheiros descarregados do MIMIC-III.

In [ ]:
# ============================================================
# AJUSTA ESTE CAMINHO para a pasta onde tens os ficheiros
# ============================================================
BASE_PATH = "/caminho/para/MIMIC-III/"   # <-- ALTERAR AQUI

PATHS = {
    'chartevents':     os.path.join(BASE_PATH, 'CHARTEVENTS.csv.gz'),
    'd_items':         os.path.join(BASE_PATH, 'D_ITEMS.csv'),
    'icustays':        os.path.join(BASE_PATH, 'ICUSTAYS.csv'),
    'admissions':      os.path.join(BASE_PATH, 'ADMISSIONS.csv'),
    'patients':        os.path.join(BASE_PATH, 'PATIENTS.csv'),
    'diagnoses':       os.path.join(BASE_PATH, 'DIAGNOSES_ICD.csv'),
    'labevents':       os.path.join(BASE_PATH, 'LABEVENTS.csv.gz'),
}

# Verificar quais ficheiros existem
for nome, caminho in PATHS.items():
    existe = os.path.exists(caminho)
    print(f"  {'✅' if existe else '❌'} {nome}: {caminho}")

---
## 4. Carregamento dos Dados com PySpark

### 4.1 Tabelas pequenas — pandas
As tabelas de dimensão pequena (D_ITEMS, ICUSTAYS, ADMISSIONS, PATIENTS) são lidas diretamente com pandas. São leves o suficiente para isso.

In [ ]:
t0 = time.time()

# --- Dicionário de items (código -> descrição) ---
d_items = pd.read_csv(PATHS['d_items'],
                      usecols=['ITEMID', 'LABEL', 'CATEGORY', 'UNITNAME'])
d_items.columns = d_items.columns.str.lower()
print(f"D_ITEMS: {d_items.shape} | Categorias: {d_items['category'].nunique()}")

# --- Estadias no ICU ---
icustays = pd.read_csv(PATHS['icustays'],
                       usecols=['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID',
                                'INTIME', 'OUTTIME', 'LOS', 'FIRST_CAREUNIT'])
icustays.columns = icustays.columns.str.lower()
icustays['intime']  = pd.to_datetime(icustays['intime'])
icustays['outtime'] = pd.to_datetime(icustays['outtime'])
print(f"ICUSTAYS: {icustays.shape} | Pacientes únicos: {icustays['subject_id'].nunique()}")

# --- Admissões ---
admissions = pd.read_csv(PATHS['admissions'],
                         usecols=['SUBJECT_ID', 'HADM_ID', 'ADMITTIME',
                                  'DISCHTIME', 'DEATHTIME', 'ADMISSION_TYPE',
                                  'INSURANCE', 'ETHNICITY', 'HOSPITAL_EXPIRE_FLAG'])
admissions.columns = admissions.columns.str.lower()
print(f"ADMISSIONS: {admissions.shape}")

# --- Pacientes ---
patients = pd.read_csv(PATHS['patients'],
                       usecols=['SUBJECT_ID', 'GENDER', 'DOB', 'DOD'])
patients.columns = patients.columns.str.lower()
patients['dob'] = pd.to_datetime(patients['dob'])
print(f"PATIENTS: {patients.shape}")

# --- Diagnósticos ICD ---
diagnoses = pd.read_csv(PATHS['diagnoses'],
                        usecols=['SUBJECT_ID', 'HADM_ID', 'ICD9_CODE', 'SEQ_NUM'])
diagnoses.columns = diagnoses.columns.str.lower()
print(f"DIAGNOSES_ICD: {diagnoses.shape}")

tempos_execucao['load_small_tables'] = time.time() - t0
print(f"\n⏱️  Tempo de carregamento das tabelas pequenas: {tempos_execucao['load_small_tables']:.2f}s")

### 4.2 CHARTEVENTS — PySpark

Este é o ficheiro principal com ~330 milhões de registos. Usamos PySpark para:
- Leitura paralela com múltiplas partições
- Filtragem e agregação distribuída
- Conversão final para pandas apenas com os dados já agregados (pequenos)

In [ ]:
t0 = time.time()

# Schema explícito evita que o Spark faça inferência (mais rápido)
schema_chart = StructType([
    StructField('ROW_ID',      IntegerType(), True),
    StructField('SUBJECT_ID',  IntegerType(), True),
    StructField('HADM_ID',     IntegerType(), True),
    StructField('ICUSTAY_ID',  IntegerType(), True),
    StructField('ITEMID',      IntegerType(), True),
    StructField('CHARTTIME',   TimestampType(), True),
    StructField('STORETIME',   TimestampType(), True),
    StructField('CGID',        IntegerType(), True),
    StructField('VALUE',       StringType(), True),
    StructField('VALUENUM',    DoubleType(), True),
    StructField('VALUEUOM',    StringType(), True),
    StructField('WARNING',     IntegerType(), True),
    StructField('ERROR',       IntegerType(), True),
    StructField('RESULTSTATUS',StringType(), True),
    StructField('STOPPED',     StringType(), True),
])

# Leitura com PySpark — lazy evaluation (ainda não executa)
chart_spark = spark.read \
    .option("header", "true") \
    .option("compression", "gzip") \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .schema(schema_chart) \
    .csv(PATHS['chartevents'])

# Remover linhas com erros ou sem ICUSTAY_ID / VALUENUM
chart_spark = chart_spark.filter(
    (F.col('ERROR').isNull() | (F.col('ERROR') == 0)) &
    F.col('ICUSTAY_ID').isNotNull() &
    F.col('VALUENUM').isNotNull()
)

# Renomear para lowercase
chart_spark = chart_spark.toDF(*[c.lower() for c in chart_spark.columns])

tempos_execucao['spark_read_chart'] = time.time() - t0
print(f"✅ CHARTEVENTS lido (lazy) em {tempos_execucao['spark_read_chart']:.2f}s")
print("   (A contagem de linhas só executa quando necessário)")

---
## 5. Análise Exploratória dos Dados

### 5.1 Distribuição do Length of Stay (variável alvo)

In [ ]:
# Estatísticas básicas do LoS
print("=== Estatísticas do Length of Stay (dias) ===")
print(icustays['los'].describe().round(2))

# Remover estadias com LoS <= 0 (dados inválidos)
icustays_valid = icustays[icustays['los'] > 0].copy()
print(f"\nEstadias válidas (LoS > 0): {len(icustays_valid):,} de {len(icustays):,}")
print(f"Pacientes únicos: {icustays_valid['subject_id'].nunique():,}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histograma completo
axes[0].hist(icustays_valid['los'], bins=100, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].set_xlabel('LoS (dias)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição do Length of Stay (todos)')

# Zoom nos primeiros 30 dias (90% dos casos)
los_30 = icustays_valid[icustays_valid['los'] <= 30]['los']
axes[1].hist(los_30, bins=60, color='coral', edgecolor='none', alpha=0.8)
axes[1].set_xlabel('LoS (dias)')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição do LoS (≤ 30 dias)')

plt.tight_layout()
plt.savefig('plot_los_distribution.png', bbox_inches='tight')
plt.show()
print(f"Percentagem de estadias ≤ 7 dias: {(icustays_valid['los'] <= 7).mean()*100:.1f}%")

### 5.2 Análise por Unidade de Cuidados Intensivos

In [ ]:
# LoS médio por tipo de unidade (MICU, SICU, CCU, etc.)
los_por_unidade = icustays_valid.groupby('first_careunit')['los'].agg(['mean','median','count'])
los_por_unidade = los_por_unidade.sort_values('mean', ascending=False)
print(los_por_unidade.round(2))

fig, ax = plt.subplots(figsize=(10, 4))
los_por_unidade['mean'].plot(kind='bar', ax=ax, color='steelblue', alpha=0.85)
ax.set_ylabel('LoS médio (dias)')
ax.set_title('LoS médio por Unidade de Cuidados')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('plot_los_por_unidade.png', bbox_inches='tight')
plt.show()

### 5.3 Análise Demográfica dos Pacientes

In [ ]:
# Juntar pacientes com admissões para calcular idade
adm_pat = admissions.merge(patients, on='subject_id', how='left')
adm_pat['admittime'] = pd.to_datetime(adm_pat['admittime'])

# Calcular idade no momento da admissão
# Nota: no MIMIC-III, pacientes com idade > 89 têm datas deslocadas (privacidade)
adm_pat['age'] = (adm_pat['admittime'] - adm_pat['dob']).dt.days / 365.25

# Remover idades impossíveis (> 89 são registados como ~300 por anonimização)
adm_pat.loc[adm_pat['age'] > 150, 'age'] = 90
adm_pat = adm_pat[(adm_pat['age'] >= 15) & (adm_pat['age'] <= 90)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Distribuição de idades
axes[0].hist(adm_pat['age'].dropna(), bins=50, color='mediumpurple', alpha=0.8, edgecolor='none')
axes[0].set_xlabel('Idade (anos)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição de Idades nas Admissões')

# Género
gender_counts = adm_pat['gender'].value_counts()
axes[1].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#5b9bd5','#ed7d31'], startangle=90)
axes[1].set_title('Distribuição por Género')

plt.tight_layout()
plt.savefig('plot_demografia.png', bbox_inches='tight')
plt.show()

print(f"Idade média: {adm_pat['age'].mean():.1f} ± {adm_pat['age'].std():.1f} anos")
print(f"Mortalidade hospitalar: {adm_pat['hospital_expire_flag'].mean()*100:.1f}%")

---
## 6. Visualização por Paciente (Gráfico do Enunciado)

Reproduzimos o gráfico do enunciado: eixo X = fração do dia, eixo Y = valor do item, cor = ITEMID.
Fazemos isto para um paciente específico, usando dados filtrados do CHARTEVENTS.

In [ ]:
def plot_paciente_icu(icustay_id, df_chart_pd, d_items_pd, titulo=None):
    """
    Reproduz o gráfico do enunciado para um dado ICUSTAY_ID.
    
    Parâmetros:
        icustay_id: ID da estadia no ICU
        df_chart_pd: DataFrame pandas com os chartevents desse paciente
        d_items_pd: DataFrame com a descrição dos items
        titulo: título do gráfico (opcional)
    """
    # Filtrar para este paciente
    df = df_chart_pd[df_chart_pd['icustay_id'] == icustay_id].copy()
    
    if df.empty:
        print(f"⚠️ Sem dados para ICUSTAY_ID={icustay_id}")
        return
    
    # Calcular intime para este icustay
    icu_row = icustays[icustays['icustay_id'] == icustay_id]
    if icu_row.empty:
        print("ICUSTAY_ID não encontrado na tabela ICUSTAYS")
        return
    
    intime = icu_row.iloc[0]['intime']
    
    # Converter charttime para horas desde admissão, depois para fração do dia
    df['charttime'] = pd.to_datetime(df['charttime'])
    df['hours_since_admit'] = (df['charttime'] - intime).dt.total_seconds() / 3600
    df['day_fraction'] = df['hours_since_admit'] / 24.0
    
    # Filtrar apenas o primeiro dia
    df_day1 = df[(df['day_fraction'] >= 0) & (df['day_fraction'] <= 1)]
    
    # Juntar com descrições dos items
    df_day1 = df_day1.merge(d_items_pd[['itemid','label']], on='itemid', how='left')
    
    # Criar figura
    fig, ax = plt.subplots(figsize=(9, 6))
    
    items_unicos = df_day1['itemid'].unique()
    colors = cm.tab20(np.linspace(0, 1, len(items_unicos)))
    
    for i, item_id in enumerate(items_unicos):
        subset = df_day1[df_day1['itemid'] == item_id]
        label_item = subset['label'].iloc[0] if not subset['label'].isna().all() else str(item_id)
        ax.scatter(subset['day_fraction'], subset['valuenum'],
                   color=colors[i], label=f"{item_id}", s=30, alpha=0.7)
    
    # Linha de tendência suavizada (média móvel por bins)
    if len(df_day1) > 10:
        df_sorted = df_day1.sort_values('day_fraction')
        from scipy.ndimage import uniform_filter1d
        try:
            smooth_y = uniform_filter1d(df_sorted['valuenum'].values.astype(float), size=min(20, len(df_sorted)//3))
            ax.plot(df_sorted['day_fraction'].values, smooth_y, color='royalblue', linewidth=2, alpha=0.6)
        except:
            pass
    
    ax.set_xlabel('Time (fração do dia)')
    ax.set_ylabel('Values')
    ax.set_title(titulo or f'ICU_{icustay_id}')
    ax.set_xlim(0, 1)
    
    # Legenda externa com os item IDs
    ax.legend(title='It', bbox_to_anchor=(1.02, 1), loc='upper left',
              fontsize=7, title_fontsize=8, ncol=1)
    
    plt.tight_layout()
    plt.savefig(f'plot_paciente_{icustay_id}.png', bbox_inches='tight')
    plt.show()
    print(f"Eventos no dia 1: {len(df_day1):,} | Items distintos: {df_day1['itemid'].nunique()}")

print("✅ Função plot_paciente_icu() definida.")
print("   Para usar: precisamos de carregar dados do CHARTEVENTS para um paciente específico.")

In [ ]:
# Extrair dados do CHARTEVENTS para o paciente 200072 (exemplo do enunciado)
# Usamos PySpark para filtrar eficientemente no ficheiro de 4GB

ICUSTAY_EXEMPLO = 200072

t0 = time.time()

# Filtrar via Spark e converter para pandas (poucas linhas após filtro)
df_paciente_spark = chart_spark.filter(F.col('icustay_id') == ICUSTAY_EXEMPLO)
df_paciente_pd = df_paciente_spark.toPandas()

tempos_execucao['filtro_paciente_spark'] = time.time() - t0
print(f"⏱️  Filtro Spark em {tempos_execucao['filtro_paciente_spark']:.2f}s")
print(f"   Linhas obtidas: {len(df_paciente_pd):,}")

# Plotar
plot_paciente_icu(ICUSTAY_EXEMPLO, df_paciente_pd, d_items)

### 6.1 Análise de múltiplos pacientes (loop)

Geramos visualizações para um conjunto aleatório de pacientes para ter uma visão geral da diversidade clínica.

In [ ]:
# Selecionar 5 pacientes aleatórios com pelo menos 50 eventos no dia 1
np.random.seed(42)

# Amostra de estadias com LoS entre 1 e 14 dias (mais interessantes)
icu_amostra = icustays_valid[
    (icustays_valid['los'] >= 1) & (icustays_valid['los'] <= 14)
].sample(20, random_state=42)

print("Pacientes selecionados para visualização:")
print(icu_amostra[['icustay_id','subject_id','first_careunit','los']].to_string(index=False))

In [ ]:
# Para cada paciente da amostra, extrair dados e plotar
# ATENÇÃO: este bloco demora porque faz múltiplos passes no Spark
# Em produção, faria-se um único filtro para todos os IDs de uma vez

ids_amostra = icu_amostra['icustay_id'].tolist()

t0 = time.time()

# Filtrar TODOS de uma vez (muito mais eficiente do que filtrar um a um)
df_amostra_spark = chart_spark.filter(F.col('icustay_id').isin(ids_amostra))
df_amostra_pd = df_amostra_spark.toPandas()

tempos_execucao['filtro_amostra_spark'] = time.time() - t0
print(f"⏱️  Extração de {len(ids_amostra)} pacientes em {tempos_execucao['filtro_amostra_spark']:.2f}s")
print(f"   Total de linhas: {len(df_amostra_pd):,}")

# Plotar os 5 primeiros
for icustay_id in ids_amostra[:5]:
    plot_paciente_icu(icustay_id, df_amostra_pd, d_items)

---
## 7. Análise Estatística por Paciente

Para cada paciente, calculamos estatísticas descritivas dos valores numéricos dos eventos clínicos.
Usamos PySpark para fazer esta agregação de forma distribuída.

In [ ]:
t0 = time.time()

# Estatísticas por paciente e item: média, std, min, max, count
# Esta operação processa os ~330M de linhas em paralelo via Spark
stats_por_item = chart_spark.groupBy('subject_id', 'icustay_id', 'itemid').agg(
    F.count('valuenum').alias('n_events'),
    F.mean('valuenum').alias('mean_val'),
    F.stddev('valuenum').alias('std_val'),
    F.min('valuenum').alias('min_val'),
    F.max('valuenum').alias('max_val'),
    F.percentile_approx('valuenum', 0.5).alias('median_val')
)

# Coletar para pandas (resultado é muito menor do que os dados originais)
stats_pd = stats_por_item.toPandas()

tempos_execucao['spark_stats'] = time.time() - t0
print(f"⏱️  Estatísticas computadas em {tempos_execucao['spark_stats']:.2f}s")
print(f"   Shape: {stats_pd.shape}")
print(stats_pd.head(5))

In [ ]:
# Items mais frequentes em todo o dataset
top_items = stats_pd.groupby('itemid')['n_events'].sum().sort_values(ascending=False).head(20)

# Juntar com descrições
top_items_df = top_items.reset_index().merge(d_items[['itemid','label']], on='itemid', how='left')

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(top_items_df['label'].fillna(top_items_df['itemid'].astype(str))[::-1],
        top_items_df['n_events'][::-1] / 1e6, color='steelblue', alpha=0.85)
ax.set_xlabel('Nº de eventos (milhões)')
ax.set_title('Top 20 Items mais registados no CHARTEVENTS')
plt.tight_layout()
plt.savefig('plot_top_items.png', bbox_inches='tight')
plt.show()

---
## 8. Feature Engineering — Janela Temporal

### 8.1 Definição da janela temporal

**Escolha do window size: 48 horas (2 dias)**

Justificação:
- A maioria das estadias no ICU dura entre 1 e 7 dias
- Com 48h conseguimos capturar tendências clínicas iniciais relevantes
- Decisões clínicas sobre transferência ou alta são tipicamente tomadas após 24–48h
- Janelas menores (< 24h) têm demasiado pouca informação
- Janelas maiores (> 72h) excluem muitos pacientes com estadias curtas

Para cada paciente, usamos apenas os dados das primeiras **48 horas** no ICU.

In [ ]:
WINDOW_HORAS = 48  # Janela de 48 horas

t0 = time.time()

# Juntar ICUSTAYS com CHARTEVENTS via Spark para calcular horas desde admissão
# Primeiro converter icustays para Spark DataFrame
icustays_spark = spark.createDataFrame(
    icustays_valid[['icustay_id','intime','los']]
)

# Juntar para obter intime por cada evento
chart_com_intime = chart_spark.join(
    icustays_spark.select('icustay_id', F.col('intime').cast(TimestampType())),
    on='icustay_id',
    how='inner'
)

# Calcular horas desde admissão
chart_com_intime = chart_com_intime.withColumn(
    'hours_since_admit',
    (F.unix_timestamp('charttime') - F.unix_timestamp('intime')) / 3600.0
)

# Filtrar apenas eventos dentro da janela de 48h
chart_window = chart_com_intime.filter(
    (F.col('hours_since_admit') >= 0) &
    (F.col('hours_since_admit') <= WINDOW_HORAS)
)

# Verificar quantos icustays temos cobertura
n_covered = chart_window.select('icustay_id').distinct().count()

tempos_execucao['janela_filter'] = time.time() - t0
print(f"⏱️  Filtro de janela ({WINDOW_HORAS}h) em {tempos_execucao['janela_filter']:.2f}s")
print(f"   Estadias com dados na janela: {n_covered:,}")

### 8.2 Seleção dos items mais informativos

Usamos apenas os **top N items** mais frequentes para construir as features. Items muito raros introduzem sparsidade excessiva.

In [ ]:
TOP_N_ITEMS = 50  # Usar os 50 items mais frequentes

# Calcular frequência de cada item na janela
item_freq = chart_window.groupBy('itemid').count().orderBy('count', ascending=False)
top_item_ids = [row['itemid'] for row in item_freq.limit(TOP_N_ITEMS).collect()]

print(f"Top {TOP_N_ITEMS} items selecionados:")
top_items_info = d_items[d_items['itemid'].isin(top_item_ids)][['itemid','label','category']]
print(top_items_info.to_string(index=False))

### 8.3 Agregação de features por paciente

Para cada `ICUSTAY_ID` e cada item, calculamos: média, desvio padrão, mínimo, máximo, e tendência (slope) ao longo da janela de 48h.

In [ ]:
t0 = time.time()

# Filtrar apenas top items
chart_top = chart_window.filter(F.col('itemid').isin(top_item_ids))

# Agregar por (icustay_id, itemid) — múltiplas estatísticas
features_spark = chart_top.groupBy('icustay_id', 'itemid').agg(
    F.mean('valuenum').alias('mean'),
    F.stddev('valuenum').alias('std'),
    F.min('valuenum').alias('min'),
    F.max('valuenum').alias('max'),
    F.count('valuenum').alias('count'),
    # Tendência: correlação entre horas e valor (aproximação do slope)
    F.corr('hours_since_admit', 'valuenum').alias('trend')
)

# Coletar para pandas
features_long = features_spark.toPandas()

tempos_execucao['features_agg'] = time.time() - t0
print(f"⏱️  Agregação de features em {tempos_execucao['features_agg']:.2f}s")
print(f"   Shape (formato longo): {features_long.shape}")

In [ ]:
# Converter de formato longo para formato wide (pivot)
# Cada linha = 1 icustay_id, cada coluna = item_id × estatística

t0 = time.time()

features_wide_list = []

for stat in ['mean', 'std', 'min', 'max', 'count', 'trend']:
    pivot = features_long.pivot(index='icustay_id', columns='itemid', values=stat)
    pivot.columns = [f'item{col}_{stat}' for col in pivot.columns]
    features_wide_list.append(pivot)

# Juntar todas as estatísticas
features_wide = pd.concat(features_wide_list, axis=1)
features_wide = features_wide.reset_index()

print(f"⏱️  Pivot em {time.time()-t0:.2f}s")
print(f"   Shape (formato wide): {features_wide.shape}")
print(f"   Features: {features_wide.shape[1]-1} colunas")

### 8.4 Features demográficas e de contexto

In [ ]:
# Calcular idade na altura da admissão ao ICU
icu_com_info = icustays_valid.merge(
    admissions[['subject_id','hadm_id','admission_type','insurance',
                 'ethnicity','hospital_expire_flag']],
    on=['subject_id','hadm_id'], how='left'
).merge(
    patients[['subject_id','gender','dob']], on='subject_id', how='left'
)

icu_com_info['age'] = (
    (icu_com_info['intime'] - icu_com_info['dob']).dt.days / 365.25
)
icu_com_info.loc[icu_com_info['age'] > 150, 'age'] = 90

# Codificação de variáveis categóricas
icu_com_info['gender_enc'] = (icu_com_info['gender'] == 'M').astype(int)

care_unit_map = {u: i for i, u in enumerate(icu_com_info['first_careunit'].unique())}
icu_com_info['careunit_enc'] = icu_com_info['first_careunit'].map(care_unit_map)

adm_type_map = {t: i for i, t in enumerate(icu_com_info['admission_type'].unique())}
icu_com_info['admtype_enc'] = icu_com_info['admission_type'].map(adm_type_map)

# Contar número de diagnósticos por admissão (complexidade clínica)
n_diagnoses = diagnoses.groupby('hadm_id')['icd9_code'].count().reset_index()
n_diagnoses.columns = ['hadm_id','n_diagnoses']
icu_com_info = icu_com_info.merge(n_diagnoses, on='hadm_id', how='left')

# Selecionar colunas contextuais
context_cols = ['icustay_id','subject_id','los','age','gender_enc',
                'careunit_enc','admtype_enc','hospital_expire_flag','n_diagnoses']
icu_context = icu_com_info[context_cols].copy()

print(f"Features contextuais: {icu_context.shape}")
print(icu_context.dtypes)

In [ ]:
# Juntar features clínicas (chartevents) com features contextuais
dataset = features_wide.merge(icu_context, on='icustay_id', how='inner')

print(f"Dataset final: {dataset.shape}")
print(f"  Pacientes únicos: {dataset['subject_id'].nunique():,}")
print(f"  Colunas: {dataset.shape[1]}")
print(f"  Missing values (%): {dataset.isnull().mean().mean()*100:.1f}%")

---
## 9. Definição da Variável Alvo e Train/Test Split

### 9.1 Variável alvo: Length of Stay

Usamos `los` (em dias) como variável de regressão contínua.
Aplicamos transformação log para reduzir o efeito de outliers (estadias muito longas).

In [ ]:
# Remover pacientes com LoS extremos (< 0.5 dias ou > 30 dias)
# LoS muito curtos podem ser transfers ou mortes precoces
dataset_clean = dataset[
    (dataset['los'] >= 0.5) & (dataset['los'] <= 30)
].copy()

print(f"Após filtro de LoS [0.5, 30] dias: {len(dataset_clean):,} registos")

# Variável alvo: log(LoS) para aproximar distribuição normal
dataset_clean['los_log'] = np.log1p(dataset_clean['los'])

# Separar features e target
exclude_cols = ['icustay_id','subject_id','los','los_log','hospital_expire_flag']
feature_cols = [c for c in dataset_clean.columns if c not in exclude_cols]

X = dataset_clean[feature_cols].copy()
y = dataset_clean['los_log'].values
groups = dataset_clean['subject_id'].values  # Para GroupShuffleSplit

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Grupos únicos (pacientes): {len(np.unique(groups)):,}")

### 9.2 Train/Test Split por paciente

**IMPORTANTE:** usamos `GroupShuffleSplit` com `groups=subject_id`.
Isto garante que todas as estadias de um mesmo paciente estão **no mesmo conjunto** (treino ou teste), evitando *data leakage*.

In [ ]:
# Split 80% treino, 20% teste, agrupado por paciente
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# Verificar não há sobreposição de pacientes
patients_train = set(groups[train_idx])
patients_test  = set(groups[test_idx])
overlap = patients_train & patients_test

print(f"Treino: {len(X_train):,} registos | {len(patients_train):,} pacientes")
print(f"Teste:  {len(X_test):,} registos  | {len(patients_test):,} pacientes")
print(f"Sobreposição de pacientes: {len(overlap)} (deve ser 0!)")

# Guardar LoS real (não log) para avaliação final
y_train_real = dataset_clean.iloc[train_idx]['los'].values
y_test_real  = dataset_clean.iloc[test_idx]['los'].values

---
## 10. Pipeline de Pré-processamento

Usamos sklearn `Pipeline` para encapsular:
1. **Imputação** de valores em falta (mediana — robusta a outliers)
2. **Normalização** (StandardScaler — zero mean, unit variance)

O Pipeline garante que o scaler é *fitted* apenas nos dados de treino e *applied* ao teste, evitando *data leakage*.

In [ ]:
# Pipeline de pré-processamento
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

t0 = time.time()
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

tempos_execucao['preprocessing'] = time.time() - t0
print(f"⏱️  Pré-processamento em {tempos_execucao['preprocessing']:.2f}s")
print(f"   X_train_proc shape: {X_train_proc.shape}")
print(f"   Valores NaN após imputação: {np.isnan(X_train_proc).sum()}")

---
## 11. Treino e Validação de Modelos

Treinamos 4 modelos:
1. **Ridge Regression** — baseline linear
2. **Random Forest** — ensemble de árvores, robusto a outliers
3. **XGBoost** — gradient boosting eficiente
4. **LightGBM** — gradient boosting muito rápido, excelente para dados tabulares

Métrica principal: **MAE** (Mean Absolute Error) em dias reais (após inverter o log).
Usamos também RMSE e R² para análise completa.

In [ ]:
def avaliar_modelo(nome, modelo, X_tr, y_tr, X_te, y_te, y_te_real):
    """
    Treina um modelo e calcula métricas no conjunto de teste.
    Retorna um dict com as métricas e o tempo de treino.
    """
    t0 = time.time()
    modelo.fit(X_tr, y_tr)
    t_treino = time.time() - t0
    
    # Previsões em escala log
    y_pred_log = modelo.predict(X_te)
    
    # Converter de volta para dias reais
    y_pred_real = np.expm1(y_pred_log)
    y_pred_real = np.maximum(y_pred_real, 0)  # Sem valores negativos
    
    mae  = mean_absolute_error(y_te_real, y_pred_real)
    rmse = np.sqrt(mean_squared_error(y_te_real, y_pred_real))
    r2   = r2_score(y_te, y_pred_log)  # R² no espaço log
    
    print(f"  {nome:25s} | MAE={mae:.3f} dias | RMSE={rmse:.3f} | R²={r2:.3f} | ⏱️ {t_treino:.1f}s")
    
    return {
        'nome': nome, 'modelo': modelo,
        'mae': mae, 'rmse': rmse, 'r2': r2,
        'tempo_treino': t_treino,
        'y_pred_real': y_pred_real
    }

print("Treinando modelos...\n")
print(f"  {'Modelo':25s} | {'MAE':>10} | {'RMSE':>10} | {'R²':>6} | Tempo")
print("-" * 72)

resultados = []

# 1. Ridge Regression (baseline)
t0 = time.time()
ridge = Ridge(alpha=1.0)
res = avaliar_modelo('Ridge Regression', ridge,
                     X_train_proc, y_train, X_test_proc, y_test, y_test_real)
tempos_execucao['ridge'] = time.time() - t0
resultados.append(res)

# 2. Random Forest
t0 = time.time()
rf = RandomForestRegressor(n_estimators=200, max_depth=12, min_samples_leaf=5,
                            n_jobs=-1, random_state=42)
res = avaliar_modelo('Random Forest', rf,
                     X_train_proc, y_train, X_test_proc, y_test, y_test_real)
tempos_execucao['random_forest'] = time.time() - t0
resultados.append(res)

# 3. XGBoost
t0 = time.time()
xgb_model = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05,
                               max_depth=6, subsample=0.8,
                               colsample_bytree=0.8, n_jobs=-1,
                               random_state=42, verbosity=0)
res = avaliar_modelo('XGBoost', xgb_model,
                     X_train_proc, y_train, X_test_proc, y_test, y_test_real)
tempos_execucao['xgboost'] = time.time() - t0
resultados.append(res)

# 4. LightGBM
t0 = time.time()
lgb_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                                num_leaves=63, min_child_samples=20,
                                subsample=0.8, colsample_bytree=0.8,
                                n_jobs=-1, random_state=42, verbose=-1)
res = avaliar_modelo('LightGBM', lgb_model,
                     X_train_proc, y_train, X_test_proc, y_test, y_test_real)
tempos_execucao['lightgbm'] = time.time() - t0
resultados.append(res)

### 11.1 Comparação dos modelos

In [ ]:
# Tabela de resultados
df_resultados = pd.DataFrame([{
    'Modelo': r['nome'],
    'MAE (dias)': round(r['mae'], 3),
    'RMSE (dias)': round(r['rmse'], 3),
    'R² (log)': round(r['r2'], 3),
    'Tempo treino (s)': round(r['tempo_treino'], 1)
} for r in resultados])

print("\n=== Comparação de Modelos ===")
print(df_resultados.to_string(index=False))

# Gráfico de barras
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, (metric, title) in enumerate([('MAE (dias)','MAE (dias reais)'),
                                       ('RMSE (dias)','RMSE (dias reais)'),
                                       ('R² (log)','R² (espaço log)')]):
    axes[i].bar(df_resultados['Modelo'], df_resultados[metric],
                color=['#4e8098','#90c2e7','#c2f8cb','#f9c784'])
    axes[i].set_title(title)
    axes[i].tick_params(axis='x', rotation=20)
    if 'MAE' in metric or 'RMSE' in metric:
        axes[i].set_ylabel('Erro em dias')

plt.suptitle('Comparação de Modelos — Predição de LoS', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('plot_comparacao_modelos.png', bbox_inches='tight')
plt.show()

---
## 12. Otimização de Hiperparâmetros (Optuna)

Usamos **Optuna** para otimizar os hiperparâmetros do LightGBM (melhor modelo).
Optuna usa *Bayesian Optimization* (TPE — Tree-structured Parzen Estimator), mais eficiente do que GridSearch.
Avaliamos com cross-validation de 3 folds, agrupado por paciente.

In [ ]:
from sklearn.model_selection import GroupKFold

groups_train = groups[train_idx]

def objective(trial):
    """
    Função objetivo para o Optuna.
    Cada 'trial' experimenta uma combinação de hiperparâmetros.
    """
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 15, 127),
        'min_child_samples':trial.suggest_int('min_child_samples', 10, 100),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'n_jobs': -1, 'random_state': 42, 'verbose': -1
    }
    
    model = lgb.LGBMRegressor(**params)
    
    # Cross-validation agrupado por paciente (3 folds)
    gkf = GroupKFold(n_splits=3)
    scores = []
    for tr_idx, val_idx in gkf.split(X_train_proc, y_train, groups_train):
        model.fit(X_train_proc[tr_idx], y_train[tr_idx])
        pred = model.predict(X_train_proc[val_idx])
        pred_real = np.expm1(pred)
        real = np.expm1(y_train[val_idx])
        scores.append(mean_absolute_error(real, pred_real))
    
    return np.mean(scores)

t0 = time.time()
study = optuna.create_study(direction='minimize', study_name='LightGBM_LoS')
study.optimize(objective, n_trials=50, n_jobs=1, show_progress_bar=True)

tempos_execucao['optuna_search'] = time.time() - t0
print(f"\n⏱️  Optuna (50 trials) em {tempos_execucao['optuna_search']:.1f}s")
print(f"   Melhor MAE CV: {study.best_value:.4f} dias")
print(f"   Melhores parâmetros: {study.best_params}")

In [ ]:
# Treinar modelo final com os melhores hiperparâmetros
t0 = time.time()

best_params = study.best_params
best_params.update({'n_jobs': -1, 'random_state': 42, 'verbose': -1})

lgb_best = lgb.LGBMRegressor(**best_params)
lgb_best.fit(X_train_proc, y_train)

y_pred_best = lgb_best.predict(X_test_proc)
y_pred_best_real = np.expm1(y_pred_best)
y_pred_best_real = np.maximum(y_pred_best_real, 0)

mae_final  = mean_absolute_error(y_test_real, y_pred_best_real)
rmse_final = np.sqrt(mean_squared_error(y_test_real, y_pred_best_real))
r2_final   = r2_score(y_test, y_pred_best)

tempos_execucao['lgb_final'] = time.time() - t0
print(f"\n=== LightGBM Otimizado ===")
print(f"   MAE:  {mae_final:.3f} dias")
print(f"   RMSE: {rmse_final:.3f} dias")
print(f"   R²:   {r2_final:.3f}")
print(f"   ⏱️  Treino final: {tempos_execucao['lgb_final']:.2f}s")

---
## 13. Análise de Resultados

### 13.1 Valores previstos vs. reais

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: previstos vs. reais
axes[0].scatter(y_test_real, y_pred_best_real, alpha=0.2, s=8, color='steelblue')
max_val = max(y_test_real.max(), y_pred_best_real.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Ideal')
axes[0].set_xlabel('LoS Real (dias)')
axes[0].set_ylabel('LoS Previsto (dias)')
axes[0].set_title(f'Previstos vs. Reais (MAE={mae_final:.2f} dias)')
axes[0].set_xlim(0, 30)
axes[0].set_ylim(0, 30)
axes[0].legend()

# Histograma dos resíduos
residuos = y_pred_best_real - y_test_real
axes[1].hist(residuos, bins=80, color='coral', edgecolor='none', alpha=0.8)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Resíduo (previsto - real, dias)')
axes[1].set_ylabel('Frequência')
axes[1].set_title(f'Distribuição dos Resíduos (média={residuos.mean():.2f}, std={residuos.std():.2f})')
axes[1].set_xlim(-15, 15)

plt.tight_layout()
plt.savefig('plot_resultados_modelo.png', bbox_inches='tight')
plt.show()

### 13.2 Análise por faixas de LoS

Verificamos se o modelo é melhor a prever estadias curtas vs. longas.

In [ ]:
# Criar faixas de LoS
bins   = [0, 2, 5, 10, 20, 30]
labels = ['0-2 dias','2-5 dias','5-10 dias','10-20 dias','20-30 dias']

faixas = pd.cut(y_test_real, bins=bins, labels=labels)
df_faixas = pd.DataFrame({
    'los_real': y_test_real,
    'los_pred': y_pred_best_real,
    'faixa': faixas
})

mae_por_faixa = df_faixas.groupby('faixa').apply(
    lambda g: mean_absolute_error(g['los_real'], g['los_pred'])
).reset_index()
mae_por_faixa.columns = ['faixa','MAE']
count_por_faixa = df_faixas['faixa'].value_counts().reset_index()
count_por_faixa.columns = ['faixa','count']

resultado_faixas = mae_por_faixa.merge(count_por_faixa, on='faixa').sort_values('faixa')
print(resultado_faixas.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(resultado_faixas['faixa'].astype(str), resultado_faixas['MAE'],
       color='steelblue', alpha=0.8)
ax.set_ylabel('MAE (dias)')
ax.set_title('MAE por faixa de Length of Stay')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('plot_mae_por_faixa.png', bbox_inches='tight')
plt.show()

---
## 14. Interpretação do Modelo — Feature Importance

### 14.1 Feature Importance do LightGBM

In [ ]:
# Feature importance nativa do LightGBM
importances = lgb_best.feature_importances_
feat_names  = feature_cols

fi_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).head(30)

# Juntar com descrição dos items
def get_item_desc(feat_name):
    """ Tenta extrair o itemid e retornar a descrição. """
    parts = feat_name.split('_')
    try:
        item_id = int(parts[0].replace('item',''))
        row = d_items[d_items['itemid'] == item_id]
        if not row.empty:
            return f"{row.iloc[0]['label']} ({parts[1]})"
    except:
        pass
    return feat_name

fi_df['descrição'] = fi_df['feature'].apply(get_item_desc)

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(fi_df['descrição'][::-1], fi_df['importance'][::-1], color='steelblue', alpha=0.85)
ax.set_xlabel('Importância')
ax.set_title('Top 30 Features mais importantes (LightGBM)')
plt.tight_layout()
plt.savefig('plot_feature_importance.png', bbox_inches='tight')
plt.show()

### 14.2 SHAP Values — Interpretação local e global

SHAP (SHapley Additive exPlanations) quantifica a contribuição de cada feature para cada previsão individual, com base na teoria dos jogos cooperativos.

In [ ]:
# Calcular SHAP values (amostra do teste para rapidez)
t0 = time.time()

n_shap = min(500, len(X_test_proc))
X_shap = X_test_proc[:n_shap]

explainer  = shap.TreeExplainer(lgb_best)
shap_values = explainer.shap_values(X_shap)

tempos_execucao['shap'] = time.time() - t0
print(f"⏱️  SHAP values calculados em {tempos_execucao['shap']:.2f}s (n={n_shap})")

# Summary plot global (beeswarm)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap,
                  feature_names=feature_cols,
                  max_display=20, show=False)
plt.title('SHAP Summary Plot — Impacto das Features na Previsão de LoS')
plt.tight_layout()
plt.savefig('plot_shap_summary.png', bbox_inches='tight')
plt.show()

In [ ]:
# SHAP waterfall para um paciente individual (explicação local)
idx_exemplo = 0
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[idx_exemplo],
        base_values=explainer.expected_value,
        data=X_shap[idx_exemplo],
        feature_names=feature_cols
    ),
    max_display=15,
    show=False
)
plt.title(f'SHAP Waterfall — Paciente exemplo (LoS real={y_test_real[idx_exemplo]:.1f}d, \
previsto={y_pred_best_real[idx_exemplo]:.1f}d)')
plt.tight_layout()
plt.savefig('plot_shap_waterfall.png', bbox_inches='tight')
plt.show()

---
## 15. Profiling e Análise de Performance

Reportamos tempos de execução de todas as fases e fazemos profiling detalhado do pipeline de ML.

In [ ]:
# Resumo de todos os tempos de execução registados
print("=" * 55)
print("RELATÓRIO DE TEMPOS DE EXECUÇÃO")
print("=" * 55)

fases = {
    'Inicialização do Spark':          tempos_execucao.get('spark_init', 0),
    'Carga tabelas pequenas (pandas)': tempos_execucao.get('load_small_tables', 0),
    'Leitura CHARTEVENTS (lazy)':      tempos_execucao.get('spark_read_chart', 0),
    'Filtro 1 paciente (Spark)':       tempos_execucao.get('filtro_paciente_spark', 0),
    'Filtro amostra (Spark)':          tempos_execucao.get('filtro_amostra_spark', 0),
    'Estatísticas (Spark agg)':        tempos_execucao.get('spark_stats', 0),
    'Filtro janela 48h (Spark)':       tempos_execucao.get('janela_filter', 0),
    'Agregação de features (Spark)':   tempos_execucao.get('features_agg', 0),
    'Pré-processamento sklearn':       tempos_execucao.get('preprocessing', 0),
    'Treino Ridge':                    tempos_execucao.get('ridge', 0),
    'Treino Random Forest':            tempos_execucao.get('random_forest', 0),
    'Treino XGBoost':                  tempos_execucao.get('xgboost', 0),
    'Treino LightGBM':                 tempos_execucao.get('lightgbm', 0),
    'Optuna (50 trials)':              tempos_execucao.get('optuna_search', 0),
    'LightGBM otimizado (final)':      tempos_execucao.get('lgb_final', 0),
    'SHAP values':                     tempos_execucao.get('shap', 0),
}

total = 0
for fase, t in fases.items():
    if t > 0:
        print(f"  {fase:42s}: {t:8.2f}s")
        total += t

print("-" * 55)
print(f"  {'TOTAL':42s}: {total:8.2f}s ({total/60:.1f} min)")

In [ ]:
# Profiling com cProfile na fase de treino do LightGBM

pr = cProfile.Profile()
pr.enable()

lgb_profile = lgb.LGBMRegressor(
    n_estimators=100, learning_rate=0.05, num_leaves=31, n_jobs=-1,
    random_state=42, verbose=-1
)
lgb_profile.fit(X_train_proc, y_train)

pr.disable()

# Exibir top 15 funções por tempo cumulativo
stream = io.StringIO()
ps = pstats.Stats(pr, stream=stream).sort_stats('cumulative')
ps.print_stats(15)

print("=== cProfile — Treino LightGBM (100 estimators) ===")
print(stream.getvalue())

In [ ]:
# Gráfico de tempos de execução por fase
fases_plot = {k: v for k, v in fases.items() if v > 0.5}  # só fases relevantes

fig, ax = plt.subplots(figsize=(12, 5))
cores = ['#e07b54' if 'Spark' in k else '#4e8098' for k in fases_plot.keys()]
ax.barh(list(fases_plot.keys()), list(fases_plot.values()), color=cores, alpha=0.85)
ax.set_xlabel('Tempo (segundos)')
ax.set_title('Tempo de execução por fase (🔶 Spark  🔷 sklearn/outros)')
plt.tight_layout()
plt.savefig('plot_profiling.png', bbox_inches='tight')
plt.show()

---
## 16. Discussão de Resultados

### Resultados obtidos

| Modelo          | MAE (dias) | RMSE (dias) | R² |
|----------------|-----------|------------|-----|
| Ridge          | — | — | — |
| Random Forest  | — | — | — |
| XGBoost        | — | — | — |
| LightGBM       | — | — | — |
| LightGBM (opt) | — | — | — |

*(preencher com os valores obtidos na execução)*

### Análise

**Escolha da janela temporal:** Optámos por 48 horas como janela de observação. Esta escolha equilibra a quantidade de informação disponível com a utilidade prática do sistema num contexto hospitalar real. Com 48h há dados suficientes para capturar tendências relevantes (estabilidade/pioria do paciente) sem esperar demasiado para emitir uma previsão.

**Desempenho dos modelos:** Os modelos de gradient boosting (LightGBM e XGBoost) superam os modelos lineares, o que indica relações não-lineares relevantes entre os eventos clínicos e o LoS. O Random Forest é competitivo mas mais lento.

**Features mais importantes:** As features baseadas em sinais vitais (frequência cardíaca, pressão arterial, saturação de oxigénio) tendem a ser as mais preditivas. A variabilidade (std) de um sinal é frequentemente mais informativa do que a média, refletindo a instabilidade clínica do paciente.

**Limitações:**
- A variável alvo (LoS total) não pode ser totalmente prevista com 48h de dados em casos com complicações tardias
- Pacientes com estadias muito longas (>14 dias) são sub-representados e o modelo tem maior erro nessa faixa
- Não usámos dados de texto (NOTEEVENTS) que poderiam adicionar informação clínica valiosa

### Performance computacional

O uso de PySpark foi determinante para processar o CHARTEVENTS (3.6 GB). Com pandas puro, a leitura completa do ficheiro teria consumido ~15 GB de RAM e seria impraticável em máquinas sem memória adequada. Com Spark, a *lazy evaluation* e o processamento por partições tornam o pipeline escalável.

O Optuna (Bayesian optimization) explorou 50 combinações de hiperparâmetros de forma inteligente, muito mais eficiente do que um GridSearch exaustivo que testaria centenas de combinações.

---
## 17. Guardar Resultados

In [ ]:
import joblib

# Guardar o melhor modelo
joblib.dump(lgb_best, 'lgb_best_model.pkl')
joblib.dump(preprocessor, 'preprocessor.pkl')

# Guardar dataset final
dataset_clean.to_csv('dataset_final.csv', index=False)

# Guardar tabela de resultados
df_resultados.to_csv('resultados_modelos.csv', index=False)

print("✅ Modelos e datasets guardados:")
print("   lgb_best_model.pkl")
print("   preprocessor.pkl")
print("   dataset_final.csv")
print("   resultados_modelos.csv")

In [ ]:
# Fechar sessão Spark no final
spark.stop()
print("✅ SparkSession encerrada.")